# WiDS Global Datathon 2026 — Submit Notebook


In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_CANDIDATES = [
    '/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26',
    '/kaggle/input/WiDSWorldWide_GlobalDathon26',
    '/kaggle/input/widsworldwide-globaldathon26',
    '.',
]
OUTPUT_PATH = '/kaggle/working/submission.csv' if os.path.exists('/kaggle/working') else 'submission.csv'
SEEDS = [42, 7, 19, 31, 57]
N_SPLITS = 5
ISO_WEIGHT = 0.30
LOGIT_WEIGHT = 1.0 - ISO_WEIGHT
TAIL_GATE_SCALE = 45.0


def locate_data_dir() -> str:
    for path in DATA_CANDIDATES:
        if os.path.exists(os.path.join(path, 'train.csv')) and os.path.exists(os.path.join(path, 'test.csv')):
            return path
    raise FileNotFoundError('Could not locate train.csv / test.csv')


def rank_against_reference(values: np.ndarray, ref_values: np.ndarray) -> np.ndarray:
    ref_sorted = np.sort(np.asarray(ref_values, dtype=float))
    if ref_sorted.size == 0:
        return np.zeros(len(values), dtype=float)
    return np.searchsorted(ref_sorted, np.asarray(values, dtype=float), side='left') / ref_sorted.size


def create_features(df: pd.DataFrame, ref_df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dist = out['dist_min_ci_0_5h'].clip(lower=1.0)
    speed = out['closing_speed_m_per_h']
    perims = out['num_perimeters_0_5h']
    area0 = out['area_first_ha'].clip(lower=0.0)

    out['dist_km'] = dist / 1000.0
    out['log_distance'] = np.log1p(dist)
    out['inv_distance'] = 1.0 / (out['dist_km'] + 0.1)
    fire_radius_m = np.sqrt(area0 * 10000.0 / np.pi)
    out['fire_radius_km'] = fire_radius_m / 1000.0
    out['area_to_dist_ratio'] = area0 / (out['dist_km'] + 0.1)
    out['log_area_dist_ratio'] = np.log1p(area0) - np.log1p(dist)
    out['has_movement'] = (perims > 1).astype(float)

    closing_pos = speed.clip(lower=0.0)
    radial_pos = out['radial_growth_rate_m_per_h'].clip(lower=0.0)
    eff_close = closing_pos + radial_pos
    out['effective_closing_speed'] = eff_close
    eta = np.where(closing_pos > 1e-3, dist / closing_pos, 9999.0)
    eta_eff = np.where(eff_close > 1e-3, dist / eff_close, 9999.0)
    out['log_eta'] = np.log1p(np.clip(eta, 0, 9999))
    out['eta_effective'] = np.clip(eta_eff, 0, 9999)
    out['log_eta_effective'] = np.log1p(out['eta_effective'])

    out['threat_score'] = out['alignment_abs'] * speed / np.log1p(dist)
    out['fire_urgency'] = perims * speed
    out['growth_intensity'] = out['area_growth_rate_ha_per_h'] * perims
    out['is_summer'] = out['event_start_month'].isin([6, 7, 8]).astype(float)
    out['is_afternoon'] = ((out['event_start_hour'] >= 12) & (out['event_start_hour'] < 20)).astype(float)

    ref_speed_near = ref_df.loc[ref_df['dist_min_ci_0_5h'] < 5000, 'closing_speed_m_per_h'].values
    out['near_speed_rank'] = 0.0
    mask_near = out['dist_min_ci_0_5h'] < 5000
    if mask_near.any():
        out.loc[mask_near, 'near_speed_rank'] = rank_against_reference(
            out.loc[mask_near, 'closing_speed_m_per_h'].values,
            ref_speed_near,
        )

    drop_cols = [
        'relative_growth_0_5h', 'projected_advance_m', 'centroid_displacement_m',
        'centroid_speed_m_per_h', 'closing_speed_abs_m_per_h', 'area_growth_abs_0_5h'
    ]
    out = out.drop(columns=[c for c in drop_cols if c in out.columns])
    out = out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return out


SCORE_FEATURES = [
    'dist_min_ci_0_5h', 'dist_km', 'log_distance', 'inv_distance', 'fire_radius_km',
    'area_to_dist_ratio', 'log_area_dist_ratio', 'closing_speed_m_per_h',
    'radial_growth_rate_m_per_h', 'effective_closing_speed', 'alignment_abs',
    'threat_score', 'dt_first_last_0_5h', 'num_perimeters_0_5h', 'has_movement',
    'area_growth_rate_ha_per_h', 'log1p_area_first', 'log1p_growth', 'dist_fit_r2_0_5h',
    'low_temporal_resolution_0_5h', 'spread_bearing_cos', 'spread_bearing_sin', 'log_eta',
    'eta_effective', 'log_eta_effective', 'fire_urgency', 'growth_intensity',
    'near_speed_rank', 'is_summer', 'is_afternoon', 'event_start_hour'
]


def build_score_models(seed: int):
    return [
        RandomForestRegressor(
            n_estimators=320,
            max_features=0.75,
            min_samples_leaf=3,
            random_state=seed,
            n_jobs=-1,
        ),
        GradientBoostingRegressor(
            random_state=seed,
            n_estimators=280,
            learning_rate=0.03,
            max_depth=2,
            subsample=0.85,
        ),
        Pipeline([
            ('scaler', StandardScaler()),
            ('ridge', Ridge(alpha=1.15)),
        ]),
    ]


def average_model_predictions(models, X_train: pd.DataFrame, y_train: np.ndarray, X_pred: pd.DataFrame) -> np.ndarray:
    pred = np.zeros(len(X_pred), dtype=float)
    for model in models:
        model.fit(X_train, y_train)
        pred += np.asarray(model.predict(X_pred), dtype=float).ravel() / len(models)
    return pred


def fit_oof_scores(X: pd.DataFrame, y: np.ndarray) -> np.ndarray:
    oof = np.zeros(len(X), dtype=float)
    counts = np.zeros(len(X), dtype=float)
    for seed in SEEDS:
        cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        for tr_idx, va_idx in cv.split(X):
            models = build_score_models(seed)
            fold_pred = average_model_predictions(models, X.iloc[tr_idx], y[tr_idx], X.iloc[va_idx])
            oof[va_idx] += fold_pred
            counts[va_idx] += 1.0
    oof /= np.maximum(counts, 1.0)
    return oof


def fit_full_scores(X: pd.DataFrame, y: np.ndarray, X_pred: pd.DataFrame) -> np.ndarray:
    full_pred = np.zeros(len(X_pred), dtype=float)
    for seed in SEEDS:
        models = build_score_models(seed)
        full_pred += average_model_predictions(models, X, y, X_pred) / len(SEEDS)
    return full_pred


def rank_percentile(values: np.ndarray) -> np.ndarray:
    order = np.argsort(values)
    ranks = np.empty(len(values), dtype=float)
    ranks[order] = np.arange(1, len(values) + 1, dtype=float) / len(values)
    return ranks


def percentile_against_reference(values: np.ndarray, ref_values: np.ndarray) -> np.ndarray:
    ref_sorted = np.sort(np.asarray(ref_values, dtype=float))
    if ref_sorted.size == 0:
        return np.zeros(len(values), dtype=float)
    return np.searchsorted(ref_sorted, np.asarray(values, dtype=float), side='right') / ref_sorted.size


def make_conditional_calibrators(risk_oof: np.ndarray, hit_times: np.ndarray):
    calibrators = {}
    x = risk_oof.reshape(-1, 1)
    for horizon in [12, 24, 48]:
        y = (hit_times <= horizon).astype(int)
        logit = LogisticRegression(max_iter=2000)
        logit.fit(x, y)
        iso = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=True, out_of_bounds='clip')
        iso.fit(risk_oof, y)
        calibrators[horizon] = (logit, iso)
    return calibrators


def apply_conditional_calibrators(calibrators, risk_values: np.ndarray) -> np.ndarray:
    out = np.zeros((len(risk_values), 3), dtype=float)
    x = risk_values.reshape(-1, 1)
    for j, horizon in enumerate([12, 24, 48]):
        logit, iso = calibrators[horizon]
        p_logit = logit.predict_proba(x)[:, 1]
        p_iso = iso.transform(risk_values)
        out[:, j] = LOGIT_WEIGHT * p_logit + ISO_WEIGHT * p_iso
    return np.clip(out, 0.0, 1.0)


def eventual_hit_gate(dist_m: np.ndarray) -> np.ndarray:
    dist_m = np.asarray(dist_m, dtype=float)
    gate = np.ones_like(dist_m, dtype=float)
    far_mask = dist_m >= 5000.0
    gate[far_mask] = expit((5000.0 - dist_m[far_mask]) / TAIL_GATE_SCALE)
    return gate


def enforce_monotonicity(p: np.ndarray) -> np.ndarray:
    p = np.clip(p, 0.0, 1.0)
    for j in range(1, p.shape[1]):
        p[:, j] = np.maximum(p[:, j], p[:, j - 1])
    return p


data_dir = locate_data_dir()
train = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test = pd.read_csv(os.path.join(data_dir, 'test.csv'))
sample = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))

train_fe = create_features(train, ref_df=train)
test_fe = create_features(test, ref_df=train)
feature_cols = [c for c in SCORE_FEATURES if c in train_fe.columns]

hit_mask = train['event'].astype(int).values == 1
hit_times = train.loc[hit_mask, 'time_to_hit_hours'].astype(float).values
X_hit = train_fe.loc[hit_mask, feature_cols].reset_index(drop=True)
y_hit = np.log1p(hit_times)

# cross-fitted score for calibration
hit_oof_score = fit_oof_scores(X_hit, y_hit)
risk_oof = rank_percentile(-hit_oof_score)
calibrators = make_conditional_calibrators(risk_oof, hit_times)

# full-train score ensemble for final inference
X_pred = pd.concat([
    train_fe.loc[hit_mask, feature_cols].reset_index(drop=True),
    test_fe[feature_cols].reset_index(drop=True),
], axis=0)
full_scores = fit_full_scores(X_hit, y_hit, X_pred)
hit_train_full_score = full_scores[:len(X_hit)]
test_full_score = full_scores[len(X_hit):]

risk_test_full = percentile_against_reference(-test_full_score, -hit_train_full_score)
cond_test = apply_conditional_calibrators(calibrators, risk_test_full)
gate_test = eventual_hit_gate(test['dist_min_ci_0_5h'].values)

preds = np.zeros((len(test), 4), dtype=float)
preds[:, 0] = gate_test * cond_test[:, 0]
preds[:, 1] = gate_test * cond_test[:, 1]
preds[:, 2] = gate_test * cond_test[:, 2]
preds[:, 3] = gate_test
preds = enforce_monotonicity(preds)

submission = pd.DataFrame({
    'event_id': test['event_id'].values,
    'prob_12h': preds[:, 0],
    'prob_24h': preds[:, 1],
    'prob_48h': preds[:, 2],
    'prob_72h': preds[:, 3],
})
submission = sample[['event_id']].merge(submission, on='event_id', how='left')

assert submission.shape[0] == sample.shape[0]
assert submission['event_id'].is_unique
assert submission['event_id'].tolist() == sample['event_id'].tolist()
assert not submission.isnull().any().any()
prob_cols = ['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']
assert ((submission[prob_cols] >= 0.0) & (submission[prob_cols] <= 1.0)).all().all()
assert (submission['prob_12h'] <= submission['prob_24h']).all()
assert (submission['prob_24h'] <= submission['prob_48h']).all()
assert (submission['prob_48h'] <= submission['prob_72h']).all()

submission.to_csv(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH}')
print(submission.head())


Saved: /kaggle/working/submission.csv
   event_id       prob_12h       prob_24h       prob_48h       prob_72h
0  10662602   0.000000e+00   0.000000e+00   0.000000e+00   0.000000e+00
1  13353600   6.722790e-01   9.199107e-01   9.681374e-01   1.000000e+00
2  13942327  1.973206e-177  3.351593e-177  3.558324e-177  4.728181e-177
3  16112781   7.068776e-01   9.266943e-01   9.718548e-01   1.000000e+00
4  17132808  2.434709e-212  2.869195e-212  3.004103e-212  3.085707e-212
